# Laboratório: RAG

**Dupla:**

- Nome 1:
- Nome 2:

Neste laboratório deverá ser montado um RAG completo sobre documentos escolhidos pela dupla. Pode ser um regulamento, um manual, uma coleção de artigos, a documentação de uma biblioteca ou qualquer conjunto de textos conhecido o bastante para julgar se uma resposta está certa. As etapas são carregar os modelos, carregar os documentos, cortar em trechos, indexar, recuperar, gerar a resposta e avaliar o resultado. Cada seção descreve o que se espera dela e traz um espaço vazio para o código, e podem ser criadas quantas células forem precisas.

A avaliação depende de um benchmark elaborado pela dupla: um conjunto de perguntas sobre os documentos, cada uma com a resposta esperada e o lugar do documento onde ela está. O benchmark deve ser escrito cedo, de preferência antes de testar o sistema, para que as perguntas não sejam escolhidas a partir do que já funciona. Ele deve incluir perguntas fáceis, perguntas cuja resposta está espalhada em mais de um trecho e pelo menos uma que os documentos não respondem.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langchain-community langchain-chroma langgraph pdfplumber beautifulsoup4
# !pip install -q langchain-groq langchain-google-genai
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import (
    DirectoryLoader,
    PDFPlumberLoader,
    TextLoader,
    WebBaseLoader,
)
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

## Modelos

Carregamento do modelo de chat que vai gerar as respostas e do modelo de embedding que vai representar os trechos e as perguntas. O modelo de embedding precisa ser o mesmo na indexação e na busca, e trocá-lo depois obriga a refazer o índice, porque o tamanho do vetor muda.

## Loading

Descrição breve dos documentos escolhidos, com a origem, o formato e o tipo de pergunta que eles respondem, seguida do carregamento de tudo como uma lista de `Document`, com o loader adequado ao formato.

### Inspeção e metadados

O texto que sai do loader deve ser inspecionado antes de seguir, porque quebras de linha, cabeçalhos repetidos, tabelas e páginas vazias afetam todas as etapas seguintes. Os metadados também devem ser conferidos e completados com o que for útil para citar a fonte ou filtrar a busca depois, como o nome do arquivo, a página corrigida ou a seção.

## Chunking

Corte dos documentos em trechos. A escolha do cortador depende da estrutura do texto, e o `chunk_size` e o `chunk_overlap` dependem do tamanho das unidades de sentido do documento, como um artigo, um parágrafo ou uma seção. A escolha deve ser justificada com base no que a inspeção mostrou.

### Análise dos trechos

Quantidade de trechos produzidos, distribuição de tamanhos e alguns exemplos. Deve ser mostrado ao menos um caso em que o corte separou uma informação que deveria ficar junta, e se a sobreposição resolveu.

## Indexação

Armazenamento dos trechos em um vector store, com os embeddings calculados pelo modelo escolhido. Os `ids` devem ser estáveis, para que rodar a célula de novo não duplique os trechos, e o índice deve ser gravado em disco, para não recalcular os embeddings a cada execução.

## Retrieval

Buscas com perguntas do benchmark, mostrando os trechos que voltam e as distâncias, e verificando se o trecho que contém a resposta aparece e em que posição.

### O valor de k e filtros

Comparação de pelo menos dois valores de `k` e, se os metadados permitirem, de uma busca com filtro. A configuração escolhida para a geração deve ser justificada.

## Geração

Prompt com uma mensagem de sistema e uma mensagem humana que receba os trechos recuperados e a pergunta, e uma função que, dada uma pergunta, recupere os trechos e devolva a resposta. A resposta deve citar a fonte de onde veio, e o modelo deve dizer que não encontrou quando os trechos não respondem, em vez de inventar.

### Teste com perguntas sem resposta

Execução da função com a pergunta do benchmark que os documentos não respondem e com uma pergunta fora do assunto, mostrando o que o modelo respondeu e se citou alguma fonte.

## Avaliação

A avaliação mede o sistema contra o benchmark, separando as duas etapas que podem falhar: a busca pode não trazer o trecho certo, e o modelo pode errar mesmo com o trecho certo no prompt.

### Benchmark

Registro do benchmark como uma lista de dicionários ou um `DataFrame`, com pelo menos dez perguntas. Cada item deve ter a pergunta, a resposta esperada e a referência da fonte, como a página ou a seção, que permite conferir se a busca acertou.

### Avaliação do retrieval

Para cada pergunta, verificação de que algum dos `k` trechos recuperados vem da fonte esperada, e cálculo da taxa de acerto sobre o benchmark inteiro.

### Avaliação das respostas

Comparação de cada resposta gerada com a resposta esperada. O julgamento pode ser feito à mão ou por um modelo como juiz com saída estruturada, que devolva um veredito e uma justificativa. Com o juiz, uma amostra dos vereditos deve ser conferida à mão.

### Análise

Tabela com a pergunta, o acerto do retrieval e o acerto da resposta, e discussão dos erros mais interessantes, indicando em qual etapa cada um aconteceu. Por fim, um parâmetro deve ser alterado, como o `chunk_size`, o `k` ou o prompt, e o benchmark rodado de novo para comparar os números.